# exp068_equivalent_pixiux_inference_port train

Train-side audit that retrains the exp063 LightGBM model family on exp063 tracker/PF/Beam output features, evaluates it with the exp039 CV surface, and saves full-train boosters for inference.

## Contents

1. Setup and configuration
2. Source artifact check
3. Exp063 model on exp039 CV
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from exp063_branch_audit import (
    EXP029_FEATURE_PATH,
    TRACKER_TRAIN_FEATURES,
    find_artifact,
    find_path,
    run_exp063_model_on_exp039_cv,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", cfg_get(config, "audit.mode"))
print("Parent:", cfg_get(config, "lineage.parent"))
print("Kernel sources:", cfg_get(config, "runtime.kaggle.kernel_sources"))


## 2. Source artifact check

In [ ]:
exp039_path = find_path(
    cfg_get(config, "data.exp039_cv_feature_path", EXP029_FEATURE_PATH),
    filename=EXP029_FEATURE_PATH.name,
)
tracker_path = find_artifact(TRACKER_TRAIN_FEATURES, cfg_get(config, "data.exp063_tracker_features_train_local"))
print("exp039 CV feature surface:", exp039_path)
print("exp063 tracker/PF/Beam train features:", tracker_path)
display(pd.read_csv(exp039_path, nrows=5))
display(pd.read_csv(tracker_path, nrows=5))


## 3. Exp063 model on exp039 CV

In [ ]:
summary = run_exp063_model_on_exp039_cv(
    output_dir=paths.artifacts_dir,
    exp039_feature_path=cfg_get(config, "data.exp039_cv_feature_path"),
    exp063_tracker_features_path=cfg_get(config, "data.exp063_tracker_features_train_local"),
    audits=tuple(cfg_get(config, "validation.audits", ["leave_one_original_fold_out", "well_hash_holdout"])),
    well_hash_folds=int(cfg_get(config, "validation.well_hash_folds", 5)),
    use_gpu=bool(cfg_get(config, "model.training.use_gpu", False)),
    fast=bool(cfg_get(config, "model.training.fast", False)),
    early_stopping_rounds=int(cfg_get(config, "model.training.early_stopping_rounds", 250)),
    max_train_rows=cfg_get(config, "model.training.max_train_rows"),
    save_full_models=bool(cfg_get(config, "model.training.save_full_models", True)),
    primary_audit_for_full_model=cfg_get(config, "model.training.primary_audit_for_full_model", "leave_one_original_fold_out"),
)
print(json.dumps(summary, indent=2))


## 4. Metrics and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "exp063_model_exp039_cv_metrics.csv")
by_well = pd.read_csv(paths.artifacts_dir / "exp063_model_exp039_cv_by_well.csv")
summary_path = paths.artifacts_dir / "exp063_model_exp039_cv_summary.json"

display(metrics)
display(by_well.head(30))
print("Summary:", summary_path, "exists=", summary_path.exists())
print("Full model manifest:", paths.artifacts_dir / "exp068_exp039_cv_full_lgb_models" / "manifest.json")
